### Run all the block one by one


In [34]:
import os, json, time
from kaggle_secrets import UserSecretsClient
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict
import google.generativeai as genai


# Setting up the API key
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
genai.configure(api_key=GOOGLE_API_KEY)
print("✅ Setup complete.")


# Hide additional warnings in the notebook
import warnings

warnings.filterwarnings("ignore")

print("✅ ADK components imported successfully.")

✅ Setup complete.
✅ ADK components imported successfully.


In [35]:
# ── MANUAL RETRY HELPER ─────────────────────────────────────────────
def generate_with_retry(model, prompt, attempts=5, initial_delay=1, multiplier=7):
    delay = initial_delay
    for attempt in range(attempts):
        try:
            response = model.generate_content(prompt)
            return response
        except Exception as e:
            if "429" in str(e) or "500" in str(e) or "503" in str(e) or "504" in str(e):
                if attempt < attempts - 1:
                    print(f"⚠️ Attempt {attempt+1} failed. Retrying in {delay}s...")
                    time.sleep(delay)
                    delay *= multiplier  # exponential backoff: 1s, 7s, 49s...
                else:
                    print("❌ All retry attempts exhausted.")
                    raise
            else:
                raise  
# ────────────────────────────────────────────────────────────────────

@dataclass
class Memory:
    messages: List[Dict] = field(default_factory=list)
    max_history: int = 20
    def add(self, role, content):
        self.messages.append({"role": role, "content": content, "time": datetime.now().isoformat()})
        if len(self.messages) > self.max_history:
            self.messages = self.messages[-self.max_history:]
    def get_context(self):
        out = ""
        for m in self.messages[-5:]:
            out += f"{m['role']}: {m['content']}\n"
        return out

#── Intent Classifier Agent ───────────────────────────────────────────────────────
class IntentAgent:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.5-flash-lite")
    def classify(self, message):
        prompt = f"""You are an intent classifier for customer support.
                Given this customer message: "{message}"
                Return ONLY a JSON object:
                - intent: one of [refund, cancellation, billing, shipping, general_help, general]
                - urgency: one of [high, medium, low]
                No explanation. No markdown. No backticks.
                Example: {{"intent": "refund", "urgency": "high"}}"""

        response = generate_with_retry(self.model, prompt)  # ← uses retry
        result = json.loads(response.text.strip())
        return result["intent"], result["urgency"]

#── Reply Agent ───────────────────────────────────────────────────────
class ReplyAgent:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.5-flash-lite")
    def create_reply(self, message, intent, urgency):
        prompt = f"""You are a friendly customer support agent.
                Customer message: "{message}"
                Detected intent: {intent}
                Urgency level: {urgency}
                Write a short, empathetic reply (2-3 sentences only).
                - Acknowledge their issue
                - Ask for ONE piece of info needed to resolve it
                - Be warm but professional
                No greetings like "Dear Customer". Just the reply."""

        response = generate_with_retry(self.model, prompt)  # ← uses retry
        return response.text.strip()

#── Coordinator Agent ───────────────────────────────────────────────────────
class Coordinator:
    def __init__(self):
        self.intent_agent = IntentAgent()
        self.reply_agent = ReplyAgent()
        self.memory = Memory()
    def ask(self, message):
        self.memory.add("user", message)
        intent, urgency = self.intent_agent.classify(message)
        reply = self.reply_agent.create_reply(message, intent, urgency)
        self.memory.add("agent", reply)
        return {"intent": intent, "urgency": urgency, "reply": reply}


print("✅ Customer Helper Agent created successfully!")
print("   Model: gemini-2.5-flash")

✅ Customer Helper Agent created successfully!
   Model: gemini-2.5-flash


#### Testing with Demo Conversations


In [ ]:
# Demo
agent = Coordinator()
messages = [
    "I want to cancel my subscription.",
    "My invoice amount is wrong.",
    #"I need a refund please.",
    #"Hello, I need help."
]
for msg in messages:
    print("USER:", msg)
    out = agent.ask(msg)
    print(json.dumps(out, indent=2))
    print("-" * 50)

USER: I want to cancel my subscription.
⚠️ Attempt 1 failed. Retrying in 1s...
⚠️ Attempt 2 failed. Retrying in 7s...
⚠️ Attempt 3 failed. Retrying in 49s...
⚠️ Attempt 4 failed. Retrying in 343s...
